# Train BPR

In [1]:
# ! pip install "pandas<=2.3.2" "numpy" "torch<=2.5" "matplotlib" "seaborn" "matplotlib-venn" "datasets" "ipykernel" "recbole" "kmeans-pytorch"

In [2]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

# Ensure logging on notebook works even on Colab
import logging
logging.getLogger().handlers.clear()

In [3]:
from typing import Any
import torch
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import BPR
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

In [4]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "beauty" 
DATA_DIR: str = "../data"
SEED = 67
DEVICE = "mps" # Other options: "cpu", "cuda"

## Create dataset

In [ ]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "cold"],
        "item": ["item_id", "cold"],
    },
    "embedding_size": 64,
    "epochs": 100,
    "train_batch_size": 1024,
    "eval_batch_size": 409_600_000,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None, 
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "seed": SEED,
}

config: Config = Config(model="BPR", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [6]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work becaus

## Train BPR

In [8]:
model: BPR = BPR(config, train_data.dataset).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

22 Jun 15:11    INFO  epoch 0 training [time: 14.30s, train loss: 755.2275]
22 Jun 15:11    INFO  epoch 0 evaluating [time: 10.78s, valid_score: 0.003100]
22 Jun 15:11    INFO  valid result: 
ndcg@20 : 0.0031    recall@20 : 0.0054    mrr@20 : 0.0038
22 Jun 15:11    INFO  Saving current: saved/BPR-Jun-22-2026_15-11-16.pth
22 Jun 15:11    INFO  epoch 1 training [time: 12.23s, train loss: 636.5695]
22 Jun 15:12    INFO  epoch 1 evaluating [time: 12.00s, valid_score: 0.005900]
22 Jun 15:12    INFO  valid result: 
ndcg@20 : 0.0059    recall@20 : 0.0104    mrr@20 : 0.0071
22 Jun 15:12    INFO  Saving current: saved/BPR-Jun-22-2026_15-11-16.pth
22 Jun 15:12    INFO  epoch 2 training [time: 12.33s, train loss: 444.7845]
22 Jun 15:12    INFO  epoch 2 evaluating [time: 11.81s, valid_score: 0.006500]
22 Jun 15:12    INFO  valid result: 
ndcg@20 : 0.0065    recall@20 : 0.0119    mrr@20 : 0.0077
22 Jun 15:12    INFO  Saving current: saved/BPR-Jun-22-2026_15-11-16.pth
22 Jun 15:12    INFO  epoch 3 t

In [9]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0068
Best valid result:
  ndcg@20: 0.0068
  recall@20: 0.0122
  mrr@20: 0.0081


## Evaluate on test set

In [10]:
test_result: dict[str, float] = trainer.evaluate(test_data, load_best_model=False)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

Test results (Overall):
  ndcg@20: 0.0062
  recall@20: 0.0112
  mrr@20: 0.0073


In [11]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    rows.append({
        "Segment": label,
        "Interactions": int(mask.sum()),
        **results,
    })

rows = []

cold_to_label = {0.0: "warm", 1.0: "cold"}

def evaluate_by_column(entity_feat, id_field, inter_id_array, entity_name):
    id_to_cold = dict(zip(
        entity_feat[id_field].numpy(),
        entity_feat["cold"].numpy(),
    ))
    for cold_val, label in cold_to_label.items():
        ids = {eid for eid, c in id_to_cold.items() if c == cold_val}
        mask = np.isin(inter_id_array, list(ids))
        if not mask.any():
            print(f"  {entity_name}-{label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"{entity_name}-{label}")

# Evaluation by user segments
evaluate_by_column(
    dataset.user_feat, dataset.uid_field,
    test_data.dataset.inter_feat[dataset.uid_field].numpy(),
    "user"
)

# Evaluation by item segments
evaluate_by_column(
    dataset.item_feat, dataset.iid_field,
    test_data.dataset.inter_feat[dataset.iid_field].numpy(),
    "item"
)

# Cross-tabulation: user × item segments
uid_to_cold = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["cold"].numpy(),
))
iid_to_cold = dict(zip(
    dataset.item_feat[dataset.iid_field].numpy(),
    dataset.item_feat["cold"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()
iid_array = test_data.dataset.inter_feat[dataset.iid_field].numpy()

for uc_val, uc_label in cold_to_label.items():
    for ic_val, ic_label in cold_to_label.items():
        uc_uids = {uid for uid, c in uid_to_cold.items() if c == uc_val}
        ic_iids = {iid for iid, c in iid_to_cold.items() if c == ic_val}
        mask = np.isin(uid_array, list(uc_uids)) & np.isin(iid_array, list(ic_iids))
        if not mask.any():
            print(f"  user-{uc_label}×item-{ic_label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"user-{uc_label}×item-{ic_label}")

# Display results sorted by NDCG
df_results = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False)
display(df_results)


/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = tor

,Segment,Interactions,ndcg@20,recall@20,mrr@20
2,item-warm,88077,0.0100,0.0179,0.0111
6,user-cold×item-warm,72341,0.0100,0.0180,0.0108
4,user-warm×item-warm,15736,0.0094,0.0168,0.0135
1,user-cold,150964,0.0069,0.0118,0.0082
0,user-warm,46742,0.0059,0.0096,0.0101
3,item-cold,109629,0.0000,0.0000,0.0000
5,user-warm×item-cold,31006,0.0000,0.0000,0.0000
7,user-cold×item-cold,78623,0.0000,0.0000,0.0000
